# core

> `VarInfo` — the one row shape every paar frontend consumes

Everything paar shows — a top-level variable, a dict entry, a DataFrame column, an
attribute — is flattened into a `VarInfo`: a plain, JSON-friendly dataclass. Two fields
carry the design:

- **`accessor`** is a *positional* path — `('data', 2, 0)` means "third child of `data`,
  then its first child", where child order is whatever the resolver yielded. Frontends send
  it back verbatim to `/expand`, `/grid`, or `/edit`; only the server ever re-walks it.
- **`more_offset`** marks the load-more sentinel row: containers are paged (`MAX_CHILDREN`
  at a time), and the sentinel tells the frontend which offset fetches the next page.

`value` is always a pre-rendered *string* (see `reprs`/`providers`) — frontends never touch
live objects, which is what makes the terminal and web UIs equally cheap.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
from dataclasses import dataclass

LAZY = '…'  # placeholder repr for values not yet evaluated (used from P1 onward)

@dataclass
class VarInfo:
    "One inspector row: a namespace binding or a container child."
    name:str
    type:str
    qualifier:str=''
    value:str=''
    shape:str|None=None
    dtype:str|None=None
    is_container:bool=False
    is_grid:bool=False
    is_error:bool=False
    path:str=''
    accessor:tuple=()
    more_offset:int|None=None   # set on the load-more sentinel: next offset to expand from

In [ ]:
from paar.core import VarInfo, LAZY
def test_varinfo_defaults():
    v = VarInfo(name='x', type='int')
    assert v.name=='x' and v.type=='int'
    assert v.qualifier=='' and v.value=='' and v.shape is None and v.dtype is None
    assert v.is_container is False and v.is_error is False and v.path==''
test_varinfo_defaults()

In [ ]:
def test_varinfo_accessor():
    v = VarInfo(name='d', type='dict', accessor=('d',))
    assert v.accessor == ('d',)
    assert VarInfo(name='x', type='int').accessor == ()   # default empty
test_varinfo_accessor()

In [ ]:
def test_varinfo_is_grid():
    assert VarInfo(name='a', type='ndarray', is_grid=True).is_grid is True
    assert VarInfo(name='x', type='int').is_grid is False
test_varinfo_is_grid()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()